# Celeb-DF-v2 Face Landmarker extraction — standalone Kaggle notebook

Notebook này chạy hoàn toàn độc lập trên Kaggle. Input là output frame/manifest đã tạo bởi notebook Celeb-DF-v2 extraction; output gồm cache MediaPipe Face Landmarker và enriched manifest cho official test set.

- Không cần clone GitHub hoặc attach source code.
- Dùng `mediapipe==1.0.0` Tasks API, không hạ NumPy/Protobuf của Kaggle.
- Tự tải model Face Landmarker chính thức (~3.6 MB) và kiểm tra SHA-256; cần bật Internet nếu chưa attach model.
- Tự tìm root chứa `frames/` và `manifests/celebdf_test.jsonl`.
- Resume theo từng video và checkpoint manifest định kỳ.
- Xử lý đúng 518 video trong official test list; quy ước nhãn giữ nguyên `0=real, 1=fake`.


In [ ]:
import importlib.metadata
import hashlib
import subprocess
import sys
import urllib.request
from pathlib import Path
from packaging.version import Version

REQUIRED_MEDIAPIPE = '1.0.0'
MIN_NUMPY = Version('2.0')
MIN_PROTOBUF = Version('5.29.5')


def installed_version(distribution):
    try:
        return Version(importlib.metadata.version(distribution))
    except importlib.metadata.PackageNotFoundError:
        return None


installed_mediapipe = installed_version('mediapipe')
installed_numpy = installed_version('numpy')
installed_protobuf = installed_version('protobuf')
environment_was_downgraded = (
    installed_numpy is not None and installed_numpy < MIN_NUMPY
) or (
    installed_protobuf is not None and installed_protobuf < MIN_PROTOBUF
)

if environment_was_downgraded:
    # Repair a session that previously installed MediaPipe 0.10.x. Binary
    # packages already loaded in this process are unsafe until a restart.
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--no-cache-dir',
        '--no-deps',
        'numpy>=2,<2.6',
        'protobuf>=5.29.5,<6',
        f'mediapipe=={REQUIRED_MEDIAPIPE}',
    ])
    raise RuntimeError(
        'Môi trường cũ đã bị hạ NumPy/Protobuf. Notebook đã sửa package; '
        'hãy chọn Session > Restart Session, sau đó Run All lần nữa.'
    )

if installed_mediapipe != Version(REQUIRED_MEDIAPIPE):
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--no-cache-dir',
        '--no-deps',
        f'mediapipe=={REQUIRED_MEDIAPIPE}',
    ])
    importlib.invalidate_caches()

import cv2
import mediapipe as mp
import numpy as np

if not hasattr(getattr(mp, 'tasks', None), 'vision'):
    raise RuntimeError('MediaPipe installation does not expose tasks.vision')

FACE_LANDMARKER_MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/face_landmarker/'
    'face_landmarker/float16/1/face_landmarker.task'
)
FACE_LANDMARKER_MODEL_SHA256 = (
    '64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff'
)
MODEL_PATH_OVERRIDE = None  # Path('/kaggle/input/.../face_landmarker.task')


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_face_landmarker_model():
    candidates = []
    if MODEL_PATH_OVERRIDE is not None:
        candidates.append(Path(MODEL_PATH_OVERRIDE))
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        candidates.extend(kaggle_input.rglob('face_landmarker.task'))
    for candidate in candidates:
        if candidate.is_file() and file_sha256(candidate) == FACE_LANDMARKER_MODEL_SHA256:
            return candidate

    target = Path('/kaggle/working/mediapipe_models/face_landmarker.task')
    if not target.is_file() or file_sha256(target) != FACE_LANDMARKER_MODEL_SHA256:
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_suffix('.download')
        print('Downloading official Face Landmarker model ...')
        urllib.request.urlretrieve(FACE_LANDMARKER_MODEL_URL, temporary)
        if file_sha256(temporary) != FACE_LANDMARKER_MODEL_SHA256:
            temporary.unlink(missing_ok=True)
            raise RuntimeError('Face Landmarker model SHA-256 mismatch')
        temporary.replace(target)
    return target


FACE_LANDMARKER_MODEL_PATH = resolve_face_landmarker_model()

print('Python    :', sys.version.split()[0])
print('NumPy     :', np.__version__)
print('OpenCV    :', cv2.__version__)
print('MediaPipe :', importlib.metadata.version('mediapipe'))
print('Tasks API :', hasattr(getattr(mp, 'tasks', None), 'vision'))
print('Model     :', FACE_LANDMARKER_MODEL_PATH)


In [ ]:
"""Portable JSONL video manifest used by extraction, landmarks, train, and evaluation."""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable


@dataclass
class VideoRecord:
    dataset: str
    split: str
    video_id: str
    label: int
    method: str
    source_video: str
    frames: list[str] = field(default_factory=list)
    source_indices: list[int] = field(default_factory=list)
    timestamps_sec: list[float] = field(default_factory=list)
    fps: float = 0.0
    landmark_path: str | None = None
    quality: dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if self.label not in (0, 1):
            raise ValueError(f"{self.video_id}: label must be 0 or 1")
        if not self.frames:
            raise ValueError(f"{self.video_id}: frame list is empty")
        if self.source_indices and len(self.source_indices) != len(self.frames):
            raise ValueError(f"{self.video_id}: source_indices/frame length mismatch")
        if self.timestamps_sec and len(self.timestamps_sec) != len(self.frames):
            raise ValueError(f"{self.video_id}: timestamps/frame length mismatch")

    def to_dict(self) -> dict[str, Any]:
        self.validate()
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> "VideoRecord":
        allowed = {item.name for item in cls.__dataclass_fields__.values()}
        record = cls(**{key: value for key, value in payload.items() if key in allowed})
        record.validate()
        return record


def write_manifest(records: Iterable[VideoRecord], path: str | Path) -> None:
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_suffix(output.suffix + ".tmp")
    count = 0
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record.to_dict(), ensure_ascii=False) + "\n")
            count += 1
    if count == 0:
        temporary.unlink(missing_ok=True)
        raise ValueError("Refusing to write an empty manifest")
    temporary.replace(output)


def load_manifest(path: str | Path) -> list[VideoRecord]:
    records: list[VideoRecord] = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                records.append(VideoRecord.from_dict(json.loads(line)))
            except Exception as error:
                raise ValueError(f"Invalid manifest line {line_number}: {error}") from error
    if not records:
        raise ValueError(f"Manifest is empty: {path}")
    return records


def manifest_summary(records: Iterable[VideoRecord]) -> dict[str, Any]:
    rows = list(records)
    by_label = {str(label): sum(row.label == label for row in rows) for label in (0, 1)}
    methods = sorted({row.method for row in rows})
    return {
        "videos": len(rows),
        "frames": sum(len(row.frames) for row in rows),
        "by_label": by_label,
        "methods": {method: sum(row.method == method for row in rows) for method in methods},
    }


import hashlib
import json
from pathlib import Path

import cv2
import numpy as np



def _landmark_fingerprint(
    record: VideoRecord,
    static_image_mode: bool,
    min_confidence: float,
    min_detected_ratio: float,
    model_sha256: str,
) -> str:
    payload = {
        "frames": record.frames,
        "source_indices": record.source_indices,
        "static_image_mode": static_image_mode,
        "min_confidence": min_confidence,
        "min_detected_ratio": min_detected_ratio,
        "model_sha256": model_sha256,
        "implementation": "mediapipe_tasks_face_landmarker_first_468_v2",
    }
    serialized = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


class FaceLandmarkerExtractor:
    def __init__(
        self,
        model_path: str | Path,
        static_image_mode: bool = True,
        min_confidence: float = 0.5,
    ) -> None:
        import mediapipe as mp

        if not static_image_mode:
            raise ValueError('Standalone extractor currently supports STATIC_IMAGE_MODE=True only')
        self.mp = mp
        options = mp.tasks.vision.FaceLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),
            running_mode=mp.tasks.vision.RunningMode.IMAGE,
            num_faces=1,
            min_face_detection_confidence=min_confidence,
            min_face_presence_confidence=min_confidence,
            min_tracking_confidence=min_confidence,
        )
        self.landmarker = mp.tasks.vision.FaceLandmarker.create_from_options(options)

    def close(self) -> None:
        self.landmarker.close()

    def process(self, image_rgb: np.ndarray) -> np.ndarray | None:
        image_rgb = np.ascontiguousarray(image_rgb, dtype=np.uint8)
        mp_image = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=image_rgb)
        result = self.landmarker.detect(mp_image)
        if not result.face_landmarks:
            return None
        points = result.face_landmarks[0]
        if len(points) < 468:
            raise RuntimeError(f'Expected at least 468 landmarks, got {len(points)}')
        # Face Landmarker outputs 478 points: first 468 are the canonical
        # face mesh and the final 10 are iris points. Keep model input stable.
        return np.asarray(
            [(point.x, point.y, point.z) for point in points[:468]],
            dtype=np.float32,
        )


def extract_landmark_cache(
    manifest_path: str | Path,
    frame_root: str | Path,
    landmark_root: str | Path,
    output_manifest: str | Path,
    model_path: str | Path,
    static_image_mode: bool = True,
    min_confidence: float = 0.5,
    min_detected_ratio: float = 0.75,
    fail_fast: bool = True,
    resume: bool = True,
    checkpoint_every: int = 25,
) -> dict[str, object]:
    from tqdm.auto import tqdm

    frame_root, landmark_root = Path(frame_root), Path(landmark_root)
    model_path = Path(model_path)
    if not model_path.is_file():
        raise FileNotFoundError(f'Face Landmarker model not found: {model_path}')
    model_sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()
    records = load_manifest(manifest_path)
    errors: list[dict[str, str]] = []
    enriched: list[VideoRecord] = []
    output_manifest = Path(output_manifest)
    input_by_key = {
        (record.dataset, record.split, record.method, record.video_id): record
        for record in records
    }
    if len(input_by_key) != len(records):
        raise ValueError("Input manifest contains duplicate dataset/split/method/video records")
    if resume and output_manifest.is_file():
        for record in load_manifest(output_manifest):
            key = (record.dataset, record.split, record.method, record.video_id)
            source_record = input_by_key.get(key)
            if (
                source_record is not None
                and record.landmark_path
                and (landmark_root / record.landmark_path).is_file()
                and record.quality.get("landmark_config_sha256")
                == _landmark_fingerprint(
                    source_record,
                    static_image_mode,
                    min_confidence,
                    min_detected_ratio,
                    model_sha256,
                )
            ):
                enriched.append(record)
    completed = {
        (record.dataset, record.split, record.method, record.video_id) for record in enriched
    }
    pending = [
        record
        for record in records
        if (record.dataset, record.split, record.method, record.video_id) not in completed
    ]
    extractor = (
        FaceLandmarkerExtractor(model_path, static_image_mode, min_confidence)
        if pending
        else None
    )
    try:
        for index, record in enumerate(tqdm(pending, desc="Extracting face landmarks"), 1):
            try:
                all_points: list[np.ndarray] = []
                detected: list[bool] = []
                image_sizes: list[tuple[int, int]] = []
                point_count = 468
                for relative_frame in record.frames:
                    image = cv2.imread(str(frame_root / relative_frame))
                    if image is None:
                        raise FileNotFoundError(f"Could not read frame: {frame_root / relative_frame}")
                    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    points = extractor.process(image_rgb)
                    image_sizes.append((image.shape[1], image.shape[0]))
                    if points is None:
                        all_points.append(np.full((point_count, 3), np.nan, dtype=np.float32))
                        detected.append(False)
                    else:
                        point_count = int(points.shape[0])
                        all_points.append(points)
                        detected.append(True)
                detected_ratio = sum(detected) / max(len(detected), 1)
                if detected_ratio < min_detected_ratio:
                    raise RuntimeError(
                        f"Face Landmarker detected only {sum(detected)}/{len(detected)} frames; "
                        f"required ratio={min_detected_ratio:.2f}"
                    )
                cache_relative = (
                    Path(record.dataset)
                    / record.split
                    / ("real" if record.label == 0 else "fake")
                    / record.method
                    / f"{record.video_id}.npz"
                )
                cache_path = landmark_root / cache_relative
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = cache_path.with_suffix(".tmp.npz")
                np.savez_compressed(
                    temporary,
                    landmarks=np.stack(all_points),
                    detected=np.asarray(detected, dtype=np.bool_),
                    image_sizes=np.asarray(image_sizes, dtype=np.int32),
                    source_indices=np.asarray(record.source_indices, dtype=np.int64),
                    timestamps_sec=np.asarray(record.timestamps_sec, dtype=np.float64),
                )
                temporary.replace(cache_path)
                record.landmark_path = str(cache_relative)
                record.quality = {
                    **record.quality,
                    "landmark_detected": int(sum(detected)),
                    "landmark_total": len(detected),
                    "landmark_detected_ratio": detected_ratio,
                    "landmark_config_sha256": _landmark_fingerprint(
                        record,
                        static_image_mode,
                        min_confidence,
                        min_detected_ratio,
                        model_sha256,
                    ),
                }
                enriched.append(record)
            except Exception as error:
                errors.append(
                    {
                        "method": record.method,
                        "class": "real" if record.label == 0 else "fake",
                        "video_id": record.video_id,
                        "error": str(error),
                    }
                )
                if fail_fast:
                    break
            if checkpoint_every > 0 and index % checkpoint_every == 0 and enriched:
                enriched.sort(key=lambda row: (row.split, row.label, row.method, row.video_id))
                write_manifest(enriched, output_manifest)
    finally:
        if extractor is not None:
            extractor.close()
    if enriched:
        enriched.sort(key=lambda row: (row.split, row.label, row.method, row.video_id))
        write_manifest(enriched, output_manifest)
    report = {
        "input_videos": len(records),
        "output_videos": len(enriched),
        "errors": errors,
        "errors_by_class": {
            class_name: sum(error["class"] == class_name for error in errors)
            for class_name in ("real", "fake")
        },
    }
    report_path = output_manifest.with_suffix(".landmarks.json")
    temporary_report = report_path.with_suffix(report_path.suffix + ".tmp")
    with temporary_report.open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)
    temporary_report.replace(report_path)
    if errors and fail_fast:
        raise RuntimeError(
            f"Landmark extraction failed for {errors[0]['video_id']}: {errors[0]['error']}"
        )
    return report


In [ ]:
from pathlib import Path

# ============================= USER CONFIG =============================
# Giữ nguyên nếu chạy trong cùng session extraction. Nếu attach output như
# Kaggle Dataset, auto-discovery bên dưới sẽ tìm root trong /kaggle/input.
FRAME_ROOT_OVERRIDE = Path('/kaggle/working/qalf_celebdf_v2_test_64f')
LANDMARK_OUTPUT_ROOT = Path('/kaggle/working/qalf_celebdf_v2_landmarks')
SPLITS_TO_RUN = ('test',)
EXPECTED_VIDEOS = {'test': 518}
EXPECTED_FRAMES_PER_VIDEO = 64

STATIC_IMAGE_MODE = True
MIN_CONFIDENCE = 0.50
MIN_DETECTED_RATIO = 0.75
CHECKPOINT_EVERY = 25
FAIL_FAST = False
RESUME = True


def is_celebdf_frame_root(path):
    path = Path(path)
    return (
        (path / 'frames').is_dir()
        and all(
            (path / 'manifests' / f'celebdf_{split}.jsonl').is_file()
            for split in SPLITS_TO_RUN
        )
    )


def find_frame_root():
    if FRAME_ROOT_OVERRIDE is not None and is_celebdf_frame_root(FRAME_ROOT_OVERRIDE):
        return Path(FRAME_ROOT_OVERRIDE)

    hits = set()
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for test_manifest in kaggle_input.rglob('celebdf_test.jsonl'):
            candidate = test_manifest.parent.parent
            if is_celebdf_frame_root(candidate):
                hits.add(candidate)

    ordered = sorted(hits, key=lambda path: (len(path.parts), str(path)))
    if not ordered:
        raise FileNotFoundError(
            'Không tìm thấy root chứa frames/ và manifests/celebdf_test.jsonl. '
            'Hãy attach dataset frame hoặc sửa FRAME_ROOT_OVERRIDE.'
        )
    if len(ordered) > 1:
        print('Có nhiều frame roots; dùng:', ordered[0])
        print('Candidates:', ordered)
    return ordered[0]


FRAME_ROOT = find_frame_root()
LANDMARK_ROOT = LANDMARK_OUTPUT_ROOT / 'landmarks'
MANIFEST_OUTPUT_ROOT = LANDMARK_OUTPUT_ROOT / 'manifests'
MANIFEST_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Frame root          :', FRAME_ROOT)
print('Face model          :', FACE_LANDMARKER_MODEL_PATH)
print('Landmark root       :', LANDMARK_ROOT)
print('Output manifest root:', MANIFEST_OUTPUT_ROOT)
print('Splits              :', SPLITS_TO_RUN)

for split in SPLITS_TO_RUN:
    manifest_path = FRAME_ROOT / 'manifests' / f'celebdf_{split}.jsonl'
    records = load_manifest(manifest_path)
    if len(records) != EXPECTED_VIDEOS[split]:
        raise RuntimeError(
            f'{split}: expected {EXPECTED_VIDEOS[split]} videos, got {len(records)}. '
            'Check that the attached dataset is the completed official Celeb-DF-v2 test output.'
        )
    invalid_frame_counts = [
        record.video_id
        for record in records
        if len(record.frames) != EXPECTED_FRAMES_PER_VIDEO
    ]
    if invalid_frame_counts:
        raise RuntimeError(
            f'{split}: {len(invalid_frame_counts)} videos do not contain '
            f'{EXPECTED_FRAMES_PER_VIDEO} frames; examples={invalid_frame_counts[:5]}'
        )
    print(
        f'{split:5s}: videos={len(records):4d}, '
        f'frames={sum(len(record.frames) for record in records):6d}'
    )


In [ ]:
all_landmark_errors = []
landmark_reports = {}

for split in SPLITS_TO_RUN:
    input_manifest = FRAME_ROOT / 'manifests' / f'celebdf_{split}.jsonl'
    output_manifest = (
        MANIFEST_OUTPUT_ROOT / f'celebdf_{split}_landmarks.jsonl'
    )

    report = extract_landmark_cache(
        manifest_path=input_manifest,
        frame_root=FRAME_ROOT,
        landmark_root=LANDMARK_ROOT,
        output_manifest=output_manifest,
        model_path=FACE_LANDMARKER_MODEL_PATH,
        static_image_mode=STATIC_IMAGE_MODE,
        min_confidence=MIN_CONFIDENCE,
        min_detected_ratio=MIN_DETECTED_RATIO,
        fail_fast=FAIL_FAST,
        resume=RESUME,
        checkpoint_every=CHECKPOINT_EVERY,
    )
    landmark_reports[split] = report
    all_landmark_errors.extend(
        {'split': split, **error}
        for error in report['errors']
    )

    print(
        f'{split:5s}: input={report["input_videos"]:4d}, '
        f'output={report["output_videos"]:4d}, '
        f'errors={len(report["errors"]):3d}'
    )

print('\nTotal landmark errors:', len(all_landmark_errors))
for error in all_landmark_errors:
    print(error)


In [ ]:
import json
from collections import Counter

qc_report = {
    'frame_root': str(FRAME_ROOT),
    'landmark_root': str(LANDMARK_ROOT),
    'face_landmarker_model': str(FACE_LANDMARKER_MODEL_PATH),
    'face_landmarker_model_sha256': FACE_LANDMARKER_MODEL_SHA256,
    'config': {
        'static_image_mode': STATIC_IMAGE_MODE,
        'min_confidence': MIN_CONFIDENCE,
        'min_detected_ratio': MIN_DETECTED_RATIO,
    },
    'splits': {},
    'errors': all_landmark_errors,
}
qc_failures = []

for split in SPLITS_TO_RUN:
    input_manifest = FRAME_ROOT / 'manifests' / f'celebdf_{split}.jsonl'
    output_manifest = (
        MANIFEST_OUTPUT_ROOT / f'celebdf_{split}_landmarks.jsonl'
    )
    input_records = load_manifest(input_manifest)
    output_records = load_manifest(output_manifest)
    errors = landmark_reports[split]['errors']

    input_keys = {
        (record.dataset, record.split, record.method, record.video_id)
        for record in input_records
    }
    output_keys = {
        (record.dataset, record.split, record.method, record.video_id)
        for record in output_records
    }

    if len(output_keys) != len(output_records):
        qc_failures.append(f'{split}: duplicate output manifest records')
    if not output_keys <= input_keys:
        qc_failures.append(f'{split}: output contains records outside input')
    if len(output_records) + len(errors) != len(input_records):
        qc_failures.append(
            f'{split}: output({len(output_records)}) + errors({len(errors)}) '
            f'!= input({len(input_records)})'
        )

    detected_ratios = []
    method_counts = Counter()
    for record in output_records:
        method_counts[record.method] += 1
        if not record.landmark_path:
            qc_failures.append(f'{split}/{record.video_id}: missing landmark_path')
            continue

        cache_path = LANDMARK_ROOT / record.landmark_path
        if not cache_path.is_file():
            qc_failures.append(f'{split}/{record.video_id}: missing {cache_path}')
            continue

        with np.load(cache_path) as cache:
            required = {
                'landmarks',
                'detected',
                'image_sizes',
                'source_indices',
                'timestamps_sec',
            }
            missing = required - set(cache.files)
            if missing:
                qc_failures.append(
                    f'{split}/{record.video_id}: missing arrays {sorted(missing)}'
                )
                continue

            landmarks = cache['landmarks']
            detected = cache['detected']
            image_sizes = cache['image_sizes']
            source_indices = cache['source_indices']
            timestamps = cache['timestamps_sec']

        frame_count = len(record.frames)
        if landmarks.shape != (frame_count, 468, 3):
            qc_failures.append(
                f'{split}/{record.video_id}: landmarks shape={landmarks.shape}'
            )
        if detected.shape != (frame_count,):
            qc_failures.append(
                f'{split}/{record.video_id}: detected shape={detected.shape}'
            )
        if image_sizes.shape != (frame_count, 2):
            qc_failures.append(
                f'{split}/{record.video_id}: image_sizes shape={image_sizes.shape}'
            )
        if source_indices.shape != (frame_count,):
            qc_failures.append(
                f'{split}/{record.video_id}: source_indices shape={source_indices.shape}'
            )
        if timestamps.shape != (frame_count,):
            qc_failures.append(
                f'{split}/{record.video_id}: timestamps shape={timestamps.shape}'
            )
        if not np.array_equal(
            source_indices,
            np.asarray(record.source_indices, dtype=np.int64),
        ):
            qc_failures.append(
                f'{split}/{record.video_id}: source indices differ from manifest'
            )

        ratio = float(np.mean(detected))
        detected_ratios.append(ratio)
        if ratio < MIN_DETECTED_RATIO:
            qc_failures.append(
                f'{split}/{record.video_id}: detected ratio={ratio:.3f}'
            )

    qc_report['splits'][split] = {
        'input_videos': len(input_records),
        'output_videos': len(output_records),
        'errors': len(errors),
        'method_counts': dict(method_counts),
        'detected_ratio': {
            'min': min(detected_ratios) if detected_ratios else None,
            'median': (
                float(np.median(detected_ratios))
                if detected_ratios else None
            ),
            'max': max(detected_ratios) if detected_ratios else None,
        },
    }

qc_report['failures'] = qc_failures
QC_REPORT_PATH = LANDMARK_OUTPUT_ROOT / 'landmark_qc_report.json'
QC_REPORT_PATH.write_text(
    json.dumps(qc_report, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print(json.dumps(qc_report['splits'], indent=2, ensure_ascii=False))
print('\nQC report:', QC_REPORT_PATH)
print('QC failures:', len(qc_failures))
for failure in qc_failures[:50]:
    print(' -', failure)

if all_landmark_errors or qc_failures:
    raise RuntimeError(
        f'Landmark extraction is incomplete: '
        f'{len(all_landmark_errors)} extraction errors, '
        f'{len(qc_failures)} QC failures. Inspect {QC_REPORT_PATH}.'
    )

print('\nLANDMARK QC PASSED')


In [ ]:
import random

import matplotlib.pyplot as plt

review_manifest = (
    MANIFEST_OUTPUT_ROOT / f'celebdf_{SPLITS_TO_RUN[0]}_landmarks.jsonl'
)
review_records = load_manifest(review_manifest)
review_records = random.Random(42).sample(
    review_records,
    min(12, len(review_records)),
)

fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for axis, record in zip(axes.ravel(), review_records):
    frame_index = len(record.frames) // 2
    image_bgr = cv2.imread(str(FRAME_ROOT / record.frames[frame_index]))
    if image_bgr is None:
        axis.set_title('IMAGE READ ERROR')
        axis.axis('off')
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    with np.load(LANDMARK_ROOT / record.landmark_path) as cache:
        points = cache['landmarks'][frame_index]
        detected = bool(cache['detected'][frame_index])

    axis.imshow(image_rgb)
    if detected and np.isfinite(points).all():
        height, width = image_rgb.shape[:2]
        sampled = points[::4]
        axis.scatter(
            sampled[:, 0] * width,
            sampled[:, 1] * height,
            s=2,
            c='lime',
            alpha=0.75,
        )

    axis.set_title(
        f'{record.method} | {record.video_id}\n'
        f'landmark_detected={detected}',
        fontsize=8,
    )
    axis.axis('off')

for axis in axes.ravel()[len(review_records):]:
    axis.axis('off')

plt.tight_layout()
plt.show()

print(
    'Visual review: landmark phải bám đúng mắt, mũi, miệng và contour; '
    'không được lệch sang background.'
)


## Output cần lưu

Lưu toàn bộ thư mục `/kaggle/working/qalf_celebdf_v2_landmarks`, gồm:

```text
landmarks/                                  # Một .npz cho mỗi video
manifests/celebdf_test_landmarks.jsonl
manifests/*.landmarks.json                  # Error reports
landmark_qc_report.json
```

Không xóa dataset frame: nhánh texture vẫn cần ảnh RGB, còn nhánh geometry đọc landmark cache.
